# Module 4.3 — Advanced Portfolio Techniques
## Beyond the Efficient Frontier

---

### On the Fragility of Optimization

If Module 4.1 taught you the **geometry** of portfolios and Module 4.2 taught you to **measure** what can go wrong, this module addresses the uncomfortable middle: **how to build portfolios that survive contact with reality**. Mean-variance optima are paper-thin—small errors in means or covariances can produce weights that are economically absurd. Transaction costs quietly tax every rebalance. Rebalancing rules that look innocent in a spreadsheet can dominate realized performance in production.

Advanced portfolio techniques are not a grab bag of tricks; they are responses to a single theme: **estimation error, path dependence, and frictions**. The Kelly criterion asks how aggressively to compound edge when the world is noisy. Hierarchical Risk Parity (HRP) asks how to diversify when correlation matrices lie. Shrinkage and robust optimization ask how to stabilize covariance before it destabilizes weights. Cost models and rebalancing policies ask how to translate a static allocation into a **process** that can be executed.

This notebook is deliberately implementation-heavy. Philosophy without code is opinion; code without philosophy is curve-fitting. We keep both.

---

### Learning Objectives

By the end of this module, you will:

1. **Derive intuition for** the Kelly criterion in single- and multi-asset settings and interpret fractional Kelly as a drawdown-control heuristic
2. **Implement** the full HRP pipeline (distance from correlation, linkage, quasi-diagonal ordering, recursive bisection)
3. **Contrast** sample minimum-variance portfolios with **shrinkage-stabilized** covariances and assess **bootstrap weight stability**
4. **Model** transaction costs with a transparent **linear + quadratic** structure in basis points of turnover
5. **Backtest** walk-forward allocations under **calendar** vs **threshold** rebalancing, comparing HRP with and without costs to shrinkage MVP

---

### Module Contents

1. **Kelly Criterion** — Geometric growth, concentration, and fractional sizing
2. **Hierarchical Risk Parity (HRP)** — Structure before optimization
3. **Robust Optimization & Shrinkage** — When the sample covariance is the enemy
4. **Transaction Cost Models** — Turnover, impact, and implementation shortfall
5. **Rebalancing Strategies** — Calendar discipline vs threshold responsiveness
6. **Integrated Practice** — Walk-forward backtests and comparisons

---

*"In the long run, survival is the only objective function that matters."*

*This module equips you to compound without courting ruin.*



In [ ]:
# ========================================
# Initial Setup and Configuration
# ========================================

from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import leaves_list as scipy_leaves_list
from scipy.cluster.hierarchy import linkage
from scipy.optimize import minimize
from scipy.spatial.distance import squareform

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

PLOT_CONFIG = {
    "figure.figsize": (14, 7),
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 100,
    "lines.linewidth": 2,
}
plt.rcParams.update(PLOT_CONFIG)

COLORS = {
    "kelly": "#C62828",
    "frac": "#1565C0",
    "hrp": "#2E7D32",
    "mvp": "#6A1B9A",
    "cost": "#F57C00",
    "neutral": "#455A64",
}

print("=" * 80)
print(" " * 12 + "MODULE 4.3: ADVANCED PORTFOLIO TECHNIQUES")
print("=" * 80)
print("✓ Environment configured successfully")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("\n📚 Ready for Kelly, HRP, costs, and rebalancing.\n")



---

## 1. Kelly Criterion — Geometric Growth Under Uncertainty

The Kelly criterion emerged from information theory and gambling, but its message is universal for quants: **maximize the expected logarithm of wealth** (equivalently, maximize geometric growth) rather than the expected level of wealth. In single-asset settings this yields a closed-form optimal fraction of capital; with multiple correlated bets, the solution couples through the covariance matrix. Full Kelly is famously aggressive—practitioners routinely use **fractional Kelly** to reduce path risk while preserving directional sizing.

### Why quants need this

Position sizing is where backtests die in production. A strategy with positive mean but poorly calibrated leverage spends most of its time in drawdowns deep enough to trigger risk limits, investor redemptions, or margin calls. Kelly (and its fractional variants) gives a principled link between **edge**, **uncertainty**, and **exposure**—and a language for debating how much optimism your covariance matrix deserves.



In [ ]:
# ========================================
# 1. Kelly: single asset, multi-asset unconstrained, fractional Kelly
# ========================================


def kelly_single(mu: float, sigma: float, rf: float = 0.0) -> float:
    """Full Kelly fraction for one risky asset (simple one-period approximation)."""
    excess = mu - rf
    if sigma <= 0:
        return 0.0
    return excess / (sigma ** 2)


def kelly_multi_unconstrained(mu: np.ndarray, cov: np.ndarray, rf: float = 0.0) -> np.ndarray:
    """Unconstrained Kelly weights (may short, may imply leverage)."""
    m = np.asarray(mu) - rf
    S = np.asarray(cov)
    w = np.linalg.solve(S, m)
    return w


def simulate_log_growth(
    returns: np.ndarray,
    w: np.ndarray,
    rf: float = 0.0,
    rf_series: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Wealth path W_t = W_{t-1} * (1 + r_p) with r_p = w'r + (1-sum|w|? cash).

    We use excess returns on risky assets; cash weight absorbs residual if |w|_1 != 1.
    """
    R = np.asarray(returns)
    T, n = R.shape
    w = np.asarray(w, dtype=float)
    w_risk = w
    invested = np.sum(w_risk)
    cash_w = 1.0 - invested
    w = np.append(w_risk, cash_w)
    if rf_series is None:
        rf_series = np.full(T, rf)
    wealth = np.ones(T + 1)
    for t in range(T):
        r_cash = rf_series[t]
        rp = float(w[:-1] @ R[t] + w[-1] * r_cash)
        wealth[t + 1] = wealth[t] * (1.0 + rp)
    return wealth


rng = np.random.RandomState(7)
n_days = 800
n_assets = 5
mu_ann = np.array([0.10, 0.08, 0.12, 0.06, 0.09])
vol_ann = np.array([0.20, 0.18, 0.28, 0.08, 0.22])
corr = np.array(
    [
        [1.0, 0.65, 0.55, 0.05, 0.50],
        [0.65, 1.0, 0.60, 0.0, 0.45],
        [0.55, 0.60, 1.0, 0.1, 0.55],
        [0.05, 0.0, 0.1, 1.0, 0.15],
        [0.50, 0.45, 0.55, 0.15, 1.0],
    ]
)
cov_ann = np.outer(vol_ann, vol_ann) * corr
mu_d = mu_ann / 252
cov_d = cov_ann / 252

Z = rng.standard_normal((n_days, n_assets))
L = np.linalg.cholesky(cov_d)
daily_rets = mu_d + Z @ L.T

rf = 0.02 / 252
mu_hat = daily_rets.mean(axis=0)
cov_hat = np.cov(daily_rets, rowvar=False)

# Single-asset Kelly on first asset
k1 = kelly_single(float(mu_hat[0]), float(np.sqrt(cov_hat[0, 0])), rf)
print(f"Single-asset Kelly (asset 0): {k1:.3f} (fraction of capital)")

# Multi unconstrained
w_k = kelly_multi_unconstrained(mu_hat, cov_hat, rf)
print("Unconstrained multi-asset Kelly weights (sum may differ from 1):")
print(pd.Series(w_k, index=[f"A{i}" for i in range(n_assets)]))

fractions = np.linspace(0.1, 1.0, 10)
terminal_log = []
for c in fractions:
    wc = c * w_k
    W = simulate_log_growth(daily_rets, wc, rf=rf)
    terminal_log.append(np.log(W[-1]))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fractions, terminal_log, color=COLORS["frac"], marker="o")
ax.set_xlabel("Fraction of full Kelly (on unconstrained vector)")
ax.set_ylabel(r"$\log W_T$ (terminal)")
ax.set_title("Fractional Kelly vs terminal log wealth (same simulated path)")
ax.axvline(1.0, color=COLORS["kelly"], linestyle="--", alpha=0.7, label="Full Kelly scale")
ax.legend()
plt.tight_layout()
plt.show()



---

## 2. Hierarchical Risk Parity (HRP)

HRP, popularized by Marcos López de Prado, reframes portfolio construction: **first recover hierarchical structure from correlations**, then allocate risk recursively along that tree. Unlike mean-variance optimization, HRP does not invert a noisy covariance matrix directly for the final weights; it uses clustering to impose **stability** and **diversification across latent factors**.

Pipeline (conceptual):

1. **Correlation matrix** $\mathbf{C}$ from past returns
2. **Distance** $d_{ij} = \sqrt{\tfrac{1}{2}(1 - \rho_{ij})}$
3. **Linkage** (e.g. single linkage) on condensed distances
4. **Quasi-diagonalization** — reorder assets so similar names sit close (we implement the standard linkage-based seriation)
5. **Recursive bisection** — split ordered blocks; allocate variance between children inversely to their cluster variances

### Why quants need this

Correlation matrices are **ill-conditioned** in high dimensions; optimizers happily exploit that ill-conditioning to produce extreme weights. HRP is a pragmatic bridge between **pure risk parity** (which ignores clusters) and **full optimization** (which overfits). For multi-asset sleeves and futures baskets, it is a standard robust baseline.



In [ ]:
# ========================================
# 2. HRP: distance, linkage, quasi-diagonal order, recursive bisection
# ========================================


def cov_to_corr(cov: np.ndarray) -> np.ndarray:
    s = np.sqrt(np.clip(np.diag(cov), 1e-12, None))
    outer = np.outer(s, s)
    return cov / outer


def distance_from_corr(corr: np.ndarray) -> np.ndarray:
    """Pairwise correlation distance (López de Prado)."""
    c = np.clip(corr, -1.0, 1.0)
    return np.sqrt(0.5 * (1.0 - c))


def quasi_diag(link: np.ndarray) -> List[int]:
    """Quasi-diagonal asset order from hierarchical linkage (dendrogram seriation).

    With ``scipy``'s ``linkage`` / ``squareform`` pipeline, ``leaves_list`` returns the
    permutation that reorders the correlation matrix into a quasi-diagonal structure
    (López de Prado's quasi-diagonalization step, via standard dendrogram leaf order).
    """
    return scipy_leaves_list(link).astype(int).tolist()


def cluster_var(cov: np.ndarray, idx: List[int]) -> float:
    """Variance of inverse-variance-weighted cluster."""
    sub = cov[np.ix_(idx, idx)]
    iv = 1.0 / np.clip(np.diag(sub), 1e-12, None)
    iv /= iv.sum()
    return float(iv @ sub @ iv)


def recursive_bisection(cov: np.ndarray, ordered_idx: List[int]) -> np.ndarray:
    """HRP weights after cov has been reordered to `ordered_idx` (asset order)."""
    n = len(ordered_idx)
    w = np.ones(n)
    clusters = [list(range(n))]
    while clusters:
        new_c = []
        for cluster in clusters:
            if len(cluster) <= 1:
                continue
            mid = len(cluster) // 2
            left = cluster[:mid]
            right = cluster[mid:]
            il = [ordered_idx[i] for i in left]
            ir = [ordered_idx[i] for i in right]
            vl = cluster_var(cov, il)
            vr = cluster_var(cov, ir)
            alpha = 1.0 - vl / (vl + vr + 1e-18)
            w[left] *= alpha
            w[right] *= 1.0 - alpha
            new_c.append(left)
            new_c.append(right)
        clusters = new_c
    return w / w.sum()


def hrp_weights(cov: np.ndarray) -> Tuple[np.ndarray, List[int], np.ndarray]:
    """Returns (weights aligned with original asset order), quasi_diag order, linkage."""
    cov = np.asarray(cov, dtype=float)
    corr = cov_to_corr(cov)
    dist = distance_from_corr(corr)
    np.fill_diagonal(dist, 0.0)
    condensed = squareform(dist, checks=False)
    link = linkage(condensed, method="single")
    od = quasi_diag(link)
    od = [int(i) for i in od]
    cov_sorted = cov[np.ix_(od, od)]
    w_sorted = recursive_bisection(cov_sorted, list(range(len(od))))
    w = np.zeros(len(od))
    w[od] = w_sorted
    return w, od, link


# Demo on same cov_hat as Kelly section (5 assets)
w_hrp, order_hrp, Zlink = hrp_weights(cov_hat)
print("Quasi-diagonal order (asset indices):", order_hrp)
print("HRP weights:")
print(pd.Series(w_hrp, index=[f"A{i}" for i in range(n_assets)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(cov_to_corr(cov_hat), cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Original correlation")
plt.colorbar(im0, ax=axes[0])
cov_q = cov_hat[np.ix_(order_hrp, order_hrp)]
im1 = axes[1].imshow(cov_to_corr(cov_q), cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Quasi-diagonalized correlation (HRP order)")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()



---

## 3. Robust Optimization — Shrinkage vs Sample MVP

Classical minimum-variance portfolios minimize $\mathbf{w}^\top \hat{\mathbf{\Sigma}} \mathbf{w}$ subject to constraints. When $\hat{\mathbf{\Sigma}}$ is **random**, the optimizer **aligns with estimation noise**—a phenomenon López de Prado likens to fitting a high-dimensional ellipsoid to fog. **Shrinkage** pulls $\hat{\mathbf{\Sigma}}$ toward a structured target (here, scaled identity) with intensity $\delta \in [0,1]$:

$$\mathbf{\Sigma}_s = (1-\delta)\hat{\mathbf{\Sigma}} + \delta \,\hat{\sigma}^2 \mathbf{I}$$

where $\hat{\sigma}^2 = \mathrm{tr}(\hat{\mathbf{\Sigma}})/N$. This is a transparent stand-in for Ledoit–Wolf–style estimators (implemented here without `sklearn`).

### Why quants need this

Without robustness machinery, **in-sample Sharpe** is often a proxy for **covariance misspecification**. Shrinkage and stability diagnostics (bootstrap distributions of weights) separate **genuine diversification** from **spurious leverage on eigenvectors**.



In [ ]:
# ========================================
# 3. Long-only min-var with shrinkage + bootstrap weight stability
# ========================================


def shrink_cov_sample(cov: np.ndarray, delta: float) -> np.ndarray:
    """Shrink sample covariance toward scaled identity (mean eigenvalue on diagonal)."""
    S = np.asarray(cov, dtype=float)
    n = S.shape[0]
    target = (np.trace(S) / n) * np.eye(n)
    d = float(np.clip(delta, 0.0, 1.0))
    return (1.0 - d) * S + d * target


def min_var_long_only(cov: np.ndarray) -> np.ndarray:
    n = cov.shape[0]
    x0 = np.ones(n) / n

    def obj(w):
        return float(w @ cov @ w)

    cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = tuple((0.0, 1.0) for _ in range(n))
    res = minimize(obj, x0, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 500})
    w = res.x.astype(float)
    return w / w.sum()


deltas = [0.0, 0.25, 0.5]
rows = []
for d in deltas:
    Ss = shrink_cov_sample(cov_hat, d)
    w = min_var_long_only(Ss)
    vol = float(np.sqrt(w @ cov_hat @ w))
    rows.append({"delta": d, "ann_vol_hat": vol * np.sqrt(252), **{f"w{i}": w[i] for i in range(n_assets)}})
mvp_compare = pd.DataFrame(rows)
print("Long-only MVP under shrinkage (annualized vol using sample cov_hat):")
print(mvp_compare)


def bootstrap_weights(
    returns: np.ndarray,
    n_boot: int = 200,
    seed: int = 0,
) -> Dict[str, np.ndarray]:
    """Bootstrap rows of returns; collect HRP and shrink-MVP weights."""
    rng = np.random.RandomState(seed)
    T, n = returns.shape
    wh, wm = [], []
    for _ in range(n_boot):
        idx = rng.randint(0, T, size=T)
        sub = returns[idx]
        cov_b = np.cov(sub, rowvar=False)
        wh.append(hrp_weights(cov_b)[0])
        wm.append(min_var_long_only(shrink_cov_sample(cov_b, 0.35)))
    return {"hrp": np.stack(wh), "mvp_shrink": np.stack(wm)}


boot = bootstrap_weights(daily_rets, n_boot=150, seed=11)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, key, title, color in zip(
    axes,
    ["hrp", "mvp_shrink"],
    ["HRP", "Long-only MVP (shrinkage δ=0.35)"],
    [COLORS["hrp"], COLORS["mvp"]],
):
    bp = ax.boxplot([boot[key][:, j] for j in range(n_assets)], patch_artist=True)
    for box in bp["boxes"]:
        box.set_facecolor(color)
        box.set_alpha(0.55)
    ax.set_xticklabels([f"A{i}" for i in range(n_assets)])
    ax.set_title(title)
    ax.set_ylabel("Weight")
plt.suptitle("Bootstrap weight stability (150 resamples of daily returns)")
plt.tight_layout()
plt.show()



---

## 4. Transaction Cost Models

Real portfolios pay **commissions**, **spreads**, and **market impact**. A minimal workable model separates a **linear** cost per unit turnover (bid-ask style) from a **quadratic** term capturing impact that grows faster than size. We express charges in **basis points of portfolio value** as a function of one-way turnover.

### Why quants need this

Without an explicit cost model, every optimizer silently assumes **infinite liquidity**. That assumption is false in small caps, in futures rolls, and whenever rebalance frequency rises. Costs transform the objective from "maximize alpha" to "maximize alpha **net of implementation**"—often reordering which strategy wins.



In [ ]:
# ========================================
# 4. CostModel: linear + quadratic bps on turnover
# ========================================


@dataclass
class CostModel:
    """Transaction cost as fraction of NAV from turnover (L1 sum of weight changes).

    total_bps = linear_bps * turnover + quad_bps * turnover**2
    cost_fraction = total_bps / 10000
    """

    linear_bps: float
    quad_bps: float

    def cost_fraction(self, turnover: float) -> float:
        t = abs(float(turnover))
        bps = self.linear_bps * t + self.quad_bps * (t ** 2)
        return bps / 10000.0


def turnover_l1(w_old: np.ndarray, w_new: np.ndarray) -> float:
    """One-way turnover: sum of absolute weight changes."""
    return float(np.sum(np.abs(w_new - w_old)))


cm = CostModel(linear_bps=8.0, quad_bps=25.0)
demo_turn = np.linspace(0, 0.6, 50)
costs = [cm.cost_fraction(t) for t in demo_turn]
plt.figure(figsize=(10, 4))
plt.plot(demo_turn, np.array(costs) * 10000, color=COLORS["cost"])
plt.xlabel("Turnover (L1 ||Δw||_1)")
plt.ylabel("Cost (bps of NAV)")
plt.title("Example CostModel: 8 bps linear + 25 bps quadratic on turnover")
plt.tight_layout()
plt.show()



---

## 5. Rebalancing Strategies

A static weight vector is not a strategy until embedded in **time**. **Calendar rebalancing** enforces discipline and predictable turnover; **threshold rebalancing** waits until drift breaches a band, trading **path dependence** for potentially lower costs. Neither dominates universally—the right rule depends on volatility, autocorrelation, and your cost surface.

### Why quants need this

Rebalancing is a **beta engine** and a **cost center**. Calendar rules are easy to audit; threshold rules interact with covariance dynamics and can cluster trades during stress. Production systems need explicit policies because implicit drift is still a policy—just an undocumented one.



---

## 6. Integrated Practice — Walk-Forward Reality Check

The cells below stitch together **rebalancing rules**, **construction engines** (HRP / MVP), and **transaction costs** inside a single walk-forward loop: at each decision date you may only use information available *before* that date. That sequencing is the difference between a flattering backtest and something you can defend in a risk review.

### Why quants need this

**Walk-forward** simulation forces your estimator, optimizer, and cost model to advance in **calendar time**—the same ordering production uses. Without it, subtle **look-ahead bias** appears: future returns leak into "past" covariance windows, rebalance timestamps align with knowledge you would not yet have, or costs are booked with information unavailable at the fill. Quants need this discipline because every unconstrained peek at the future quietly **monetizes fake alpha**; explicit walk-forward boundaries are how you catch it before capital does.



In [ ]:
# ========================================
# 6. Walk-forward backtest: monthly vs threshold; HRP / MVP; costs on/off
# ========================================


def backtest_weights(
    returns: pd.DataFrame,
    weight_fn: Callable[[np.ndarray], np.ndarray],
    train: int = 252,
    mode: str = "monthly",
    threshold: float = 0.06,
    costs: Optional[CostModel] = None,
) -> pd.Series:
    """Walk-forward: at rebalance dates, estimate cov on past `train` days, set weights.

    mode: 'monthly' (first day of month after burn-in) or 'threshold' on max |w - w_target|.
    For threshold mode, each day we compute *hypothetical* weights from the rolling window;
    if sup-norm vs current held weights exceeds `threshold`, we rebalance (pay costs, update w).
    """
    R = returns.values
    idx = returns.index
    T, n = R.shape
    w = np.ones(n) / n
    wealth = np.ones(T)
    for t in range(1, T):
        if t < train:
            rp = float(w @ R[t])
            wealth[t] = wealth[t - 1] * (1.0 + rp)
            continue

        window = R[t - train : t]
        cov_est = np.cov(window, rowvar=False)

        if mode == "monthly":
            rebalance = (t == train) or (idx[t].month != idx[t - 1].month)
        elif mode == "threshold":
            w_hyp = weight_fn(cov_est)
            rebalance = (t == train) or (float(np.max(np.abs(w_hyp - w))) >= threshold)
        else:
            raise ValueError("mode must be 'monthly' or 'threshold'")

        if rebalance:
            w_new = weight_fn(cov_est)
            if costs is not None:
                tor = turnover_l1(w, w_new)
                wealth[t - 1] *= 1.0 - costs.cost_fraction(tor)
            w = w_new

        rp = float(w @ R[t])
        wealth[t] = wealth[t - 1] * (1.0 + rp)

    return pd.Series(wealth, index=idx, name="wealth")


# Richer synthetic panel for backtest (8 assets)
names8 = ["EQ1", "EQ2", "EQ3", "BD1", "BD2", "REIT", "CMDTY", "GLD"]
rng_bt = np.random.RandomState(RANDOM_SEED)
n8 = len(names8)
mu8 = np.array([0.09, 0.08, 0.11, 0.03, 0.04, 0.085, 0.05, 0.045])
vol8 = np.array([0.18, 0.20, 0.26, 0.05, 0.07, 0.22, 0.19, 0.16])
c8 = np.eye(n8)
for i in range(n8):
    for j in range(i + 1, n8):
        if i < 3 and j < 3:
            c8[i, j] = c8[j, i] = 0.72
        elif i >= 3 and j >= 3 and i < 5 and j < 5:
            c8[i, j] = c8[j, i] = 0.75
        else:
            c8[i, j] = c8[j, i] = rng_bt.uniform(0.05, 0.35)
cov8 = np.outer(vol8, vol8) * c8
L8 = np.linalg.cholesky(cov8 / 252)
T_bt = 1400
Z8 = rng_bt.standard_normal((T_bt, n8))
rets8 = mu8 / 252 + Z8 @ L8.T
dates8 = pd.bdate_range("2018-01-02", periods=T_bt)
df8 = pd.DataFrame(rets8, index=dates8, columns=names8)

delta_mvp = 0.35
cost_model = CostModel(linear_bps=10.0, quad_bps=40.0)


def w_hrp_fn(cov: np.ndarray) -> np.ndarray:
    return hrp_weights(cov)[0]


def w_mvp_fn(cov: np.ndarray) -> np.ndarray:
    return min_var_long_only(shrink_cov_sample(cov, delta_mvp))


results = {}
for mode in ("monthly", "threshold"):
    results[(mode, "hrp", "no_cost")] = backtest_weights(df8, w_hrp_fn, train=252, mode=mode, threshold=0.07, costs=None)
    results[(mode, "hrp", "cost")] = backtest_weights(df8, w_hrp_fn, train=252, mode=mode, threshold=0.07, costs=cost_model)
    results[(mode, "mvp", "cost")] = backtest_weights(df8, w_mvp_fn, train=252, mode=mode, threshold=0.07, costs=cost_model)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
plot_specs = [
    ("hrp", "no_cost", "-", COLORS["hrp"], "HRP (no costs)"),
    ("hrp", "cost", "--", COLORS["hrp"], "HRP + costs"),
    ("mvp", "cost", "-.", COLORS["mvp"], "MVP shrink + costs"),
]
for i, mode in enumerate(["monthly", "threshold"]):
    ax = axes[i]
    for model, cst, ls, clr, lbl in plot_specs:
        s = results[(mode, model, cst)]
        ax.plot(s.index, s.values / s.iloc[0], linestyle=ls, color=clr, label=lbl, linewidth=2)
    ax.set_title(f"Walk-forward wealth (normalized) — {mode} rebalancing")
    ax.set_ylabel("Wealth")
    ax.legend(loc="upper left")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

summary = []
for k, s in results.items():
    mode, model, cst = k
    cr = float(s.iloc[-1] / s.iloc[0])
    ann = cr ** (252 / len(s)) - 1
    summary.append({"mode": mode, "model": model, "costs": cst, "total_return": cr - 1, "ann_approx": ann})
print(pd.DataFrame(summary).sort_values(["mode", "model", "costs"]).reset_index(drop=True))



---

## Closing Thought

Kelly teaches **how large** to bet when the model is wrong in well-understood ways. HRP and shrinkage teach **how not to bet** on eigenvectors of noise. Cost models and rebalance rules teach **how to stay solvent** while doing both. Together, they are the difference between a portfolio that exists on a whiteboard and one that survives compounding.

---

*End of Module 4.3*

